# 03 Calibration And CI

Generated by `scripts/revision/create_revision_notebooks.py`.


## Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = 'revision/colab-pipeline'
REPO_DIR = DRIVE_ROOT / 'ECG-RAMBA'

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)

def _run_setup(cmd, cwd=None, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, cwd=str(cwd) if cwd else None, check=check)

if (REPO_DIR / '.git').exists():
    os.chdir(REPO_DIR)
    _run_setup('git fetch origin')
    current_branch = subprocess.check_output(
        'git branch --show-current',
        shell=True,
        text=True,
    ).strip()
    if current_branch != BRANCH:
        _run_setup(f'git checkout {BRANCH}')
    _run_setup(f'git pull --ff-only --autostash origin {BRANCH}')
elif (REPO_DIR / 'configs' / 'config.py').exists():
    os.chdir(REPO_DIR)
    print('Repo directory exists but is not a git checkout; skipping git pull.')
elif Path.cwd().joinpath('configs', 'config.py').exists():
    REPO_DIR = Path.cwd()
    os.chdir(REPO_DIR)
    if (REPO_DIR / '.git').exists():
        _run_setup('git fetch origin')
        _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', check=False)
else:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    _run_setup(f'git clone -b {BRANCH} {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)

os.chdir(REPO_DIR)
_run_setup('git status --short --branch', check=False)

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)

def run(cmd, check=True):
    print(f'$ {cmd}')
    return subprocess.run(cmd, shell=True, check=check)


def run_script_if_exists(script_path, command):
    path = Path(script_path)
    if path.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')


## Run Calibration And Bootstrap CI

In [ ]:
pred_dir = Path('reports/revision/predictions')
metric_dir = Path('reports/revision/metrics')
metric_dir.mkdir(parents=True, exist_ok=True)

prediction_files = sorted(pred_dir.glob('*.npz'))
if not prediction_files:
    print(f'No prediction NPZ files found under {pred_dir}. Run notebook 02 first.')

for pred in prediction_files:
    out = metric_dir / f'calibration_ci_{pred.stem}.json'
    cmd = f'python scripts/revision/04_calibration_ci.py --predictions {pred} --out {out} --n-boot 1000'
    run(cmd)


## Summarize Metric Files

In [ ]:
for path in sorted(Path('reports/revision/metrics').glob('calibration_ci_*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    print('\n' + str(path))
    print('shape:', payload.get('shape'))
    print('metrics:', payload.get('metrics'))
    print('calibration:', payload.get('calibration'))


In [ ]:
!python scripts/revision/05_artifact_inventory.py
